# Анализ снижения активности клиентов банка

## 1. Постановка задачи

Банк «Меридиан» хочет вернуть карточную активность клиентов. Исследование должно определить, у кого она снизилась, с какими наблюдаемыми характеристиками связано снижение и каким группам можно предложить отдельную коммуникацию. Банк вымышленный, данные синтетические; выводы предназначены руководителю розничных продуктов.

Итог всей работы — 3–5 объяснимых групп по правилу «кто клиент + что произошло с его активностью». В каждой группе не менее 150 клиентов, во всей кампании не более 2000 уникальных клиентов; итоговые группы не пересекаются. Обязательно согласие `marketing_opt_in`, причины исключения и возможность повторить правила на новой выгрузке. Метрику выбираем позднее, рассчитываем в пересчёте на месяц и объясняем, какое снижение она не видит.

| Источник / период | Границы | Смысл |
|---|---|---|
| Базовый период | 01.01.2024–31.12.2024 | 12 месяцев |
| Сравниваемый период | 01.01.2025–30.06.2025 | 6 месяцев |
| Справочник клиентов | Снимок на 30.06.2025 | Действующие клиенты по описанию источника |

Суммарные показатели за 12 и 6 месяцев напрямую несопоставимы. Строка `clients.csv` — запись о клиенте с предполагаемым ключом `client_id`; строка `transactions.csv` — запись об операции с предполагаемым ключом `txn_id`. Общая ссылка — `client_id`, ожидаемая связь 1:N требует проверки. В дальнейшем единицей сравнения станет клиент-месяц; строки справочника, уникальные клиенты и операции считаются раздельно.

Дальнейшее исследование отвечает на пять вопросов:

1. Какая месячная метрика отражает использование карт и какие изменения она не замечает?
2. У каких клиентов и в каких наблюдаемых группах активность снизилась?
3. С какими категориями операций и каналами связано изменение?
4. Не связано ли видимое снижение с переходом в другой канал или с сезонностью?
5. Какие группы можно включить в кампанию с учётом согласия, размера групп и общего лимита?

Наблюдаемая связь признака со снижением ещё не доказывает причину изменения или мотив клиента. Эти вопросы будут проверяться после подготовки данных; заранее выбирать группы и объяснения нельзя.


### Словарь данных: клиенты

| Поле | Бизнес-смысл | Логическая шкала / единицы | Фактическое представление |
|---|---|---|---|
| `client_id` | Идентификатор клиента | Ключ, не количественная мера | `C003407`; 5668 уникальных ID, есть повторы |
| `birth_date` | Дата рождения | Дата, интервальная шкала | `YYYY-MM-DD` |
| `gender` | Пол | Номинальная | `М`, `Ж` |
| `city` | Город проживания | Номинальная | 17 исходных написаний; есть регистр, сокращения и пробелы |
| `segment` | Сегмент клиента | Номинальная | Массовый, Зарплатный, Премиальный, Private |
| `loyalty_tier` | Уровень лояльности | Порядковая | Базовый → Серебро → Золото → Платина |
| `card_type` | Тип текущей карты | Номинальная | Дебетовая, Кредитная, Премиальная |
| `registration_date` | Дата прихода в банк | Дата, интервальная шкала | `DD.MM.YYYY` |
| `card_open_date` | Дата открытия текущей карты | Дата, интервальная шкала | `DD.MM.YYYY` |
| `income_source` | Заявленный источник зарплаты | Номинальная | Зарплата в нашем банке / в другом банке / Не указан |
| `monthly_income` | Месячный доход | Количественная, шкала отношений; рубли | Текст с десятичной точкой, например `108600.0`; есть пустоты |
| `has_mobile_app` | Установлено ли приложение | Бинарная номинальная | `Да`, `Нет` |
| `app_last_login` | Месяц последнего входа | Временной признак с точностью до месяца | `YYYY-MM`; есть пустоты, точный день неизвестен |
| `satisfaction_score` | Оценка сервиса в опросе | Порядковая, 1–5 | `1`, `2`, `3`, `4`, `5`; есть пустоты; автоматически не усредняем |
| `branch_code` | Код отделения | Идентификатор | Например `00395`; 839 кодов, ведущие нули сохранены |
| `marketing_opt_in` | Согласие на коммуникацию | Бинарная номинальная | `Да`, `Нет` |

### Словарь данных: операции

| Поле | Бизнес-смысл | Логическая шкала / единицы | Фактическое представление |
|---|---|---|---|
| `txn_id` | Идентификатор операции | Ключ, не количественная мера | Текст цифр, например `990023`; уникальность нарушена |
| `client_id` | Клиент операции | Ссылка на справочник | Например `C005169`; 5990 уникальных ID |
| `txn_datetime` | Дата и время операции | Время, интервальная шкала | `YYYY-MM-DD HH:MM:SS` и `YYYY-MM-DD`; время есть не у всех |
| `amount` | Сумма операции | Количественная, шкала отношений; валюта `currency` | Например `1430,95`; десятичная запятая сохранена |
| `currency` | Валюта операции | Номинальная | `RUB`, `EUR`, `USD`; без конвертации суммы не складываем |
| `category` | Категория траты | Номинальная | 12 написаний при 9 категориях в описании; полный домен в частотах |
| `mcc_code` | Код типа торговой точки | Номинальный код, не мера | 9 кодов: 4111, 4814, 5411, 5541, 5651, 5812, 5815, 5912, 7832 |
| `channel` | Способ оплаты | Номинальная | Карта, Токен, СБП, Онлайн-банк |
| `status` | Результат операции | Номинальная | успешно, отклонено, возврат |

«Карта» — оплата пластиком; «Токен» — смартфоном или часами; «СБП» — перевод через Систему быстрых платежей; «Онлайн-банк» — операция в интернет-банке. Связь `category` и `mcc_code` проверена ниже: в 1569 исходных строках пары отличаются от преобладающего соответствия в этой выгрузке. Это основание для проверки, но правильность одного из двух полей отдельно не установлена.

## 2. Аудит исходных данных

### 2.1. Среда и источники

In [1]:
import sys
import hashlib
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
project_root = Path.cwd()
raw_dir = project_root / "data" / "raw"
required_files = [project_root / "pyproject.toml", raw_dir / "clients.csv", raw_dir / "transactions.csv"]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Запустите ноутбук из корня проекта. Не найдены: " + ", ".join(missing_files))

print("Python:", sys.executable)
print("Версия Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Проект:", project_root)
print("Python из .venv проекта:", Path(sys.executable).resolve().parent == project_root / ".venv" / "Scripts")

Python: D:\DS_SBER\LAB_1\.venv\Scripts\python.exe
Версия Python: 3.14.4
pandas: 3.1.0
Проект: D:\DS_SBER\LAB_1
Python из .venv проекта: True


In [3]:
source_hashes = {}
for filename in ["clients.csv", "transactions.csv"]:
    path = raw_dir / filename
    source_hashes[filename] = hashlib.sha256(path.read_bytes()).hexdigest()
    print(filename, path.stat().st_size, "байт")
    print("SHA-256:", source_hashes[filename])

clients.csv 1061740 байт
SHA-256: d778cabc9a4d65de6aa6cfcbcf4c57d612129ffd5f8091a170d37b19429b0bbe
transactions.csv 47659568 байт
SHA-256: ad3a4347cd163d8d6e08da12ff6b2182b42fdf006b7db0bd6bd5a03acd9b2e7d


Здесь проверили, из какой папки запускается ноутбук и какой Python выполняет его код. Это нужно, чтобы случайно не прочитать другие файлы с такими же названиями и не использовать окружение другого проекта. Код сначала ищет конфигурацию проекта и оба исходных CSV, после чего показывает путь к интерпретатору. Если обязательного файла нет, выполнение останавливается и сообщает, что нужно проверить рабочую папку.

Ноутбук выполняется из `D:\DS_SBER\LAB_1`, используется Python 3.14.4 из окружения `.venv` этого проекта. Размер `clients.csv` составляет 1 061 740 байт, размер `transactions.csv` — 47 659 568 байт. Для каждого файла также рассчитали SHA-256 — контрольный отпечаток его содержимого. В конце блока этот отпечаток будет рассчитан повторно, чтобы проверить, сохранились ли исходные файлы без изменений.

### 2.2. Форматы CSV и загрузка

In [4]:
for filename, encoding in [("clients.csv", "utf-8-sig"), ("transactions.csv", "cp1251")]:
    path = raw_dir / filename
    with path.open("rb") as file:
        sample = file.read(300)
    print(f"\nФайл: {filename}")
    print(repr(sample))
    with path.open(encoding=encoding) as file:
        for _ in range(3):
            print(file.readline().rstrip("\n"))


Файл: clients.csv
b'client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in\nC003407,1964-09-12,\xd0\x9c,\xd0\x9c\xd0\xbe\xd1\x81\xd0\xba\xd0\xb2\xd0\xb0,\xd0\x9f\xd1\x80\xd0\xb5\xd0\xbc\xd0\xb8\xd0\xb0\xd0\xbb\xd1\x8c\xd0\xbd\xd1\x8b\xd0\xb9,\xd0\xa1\xd0\xb5\xd1\x80\xd0\xb5\xd0\xb1\xd1\x80\xd0\xbe,\xd0\x9f\xd1\x80\xd0\xb5\xd0\xbc\xd0\xb8\xd0\xb0\xd0\xbb\xd1\x8c\xd0\xbd\xd0\xb0\xd1\x8f,'
client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
C003407,1964-09-12,М,Москва,Премиальный,Серебро,Премиальная,20.06.2020,20.06.2022,Зарплата в нашем банке,108600.0,Да,2024-12,5,00395,Да
C003983,1965-08-21,М,Новосибирск,Массовый,Базовый,Кредитная,23.07.2016,30.07.2016,Не указан,65200.0,Да,2024-01,,00342,Да

Перед загрузкой посмотрели, как данные записаны непосредственно в файлах. Сначала прочитали первые 300 байт, затем показали первые три строки каждого файла в подходящей кодировке. Такой просмотр помогает определить разделитель столбцов и формат значений до того, как pandas попытается собрать из них таблицу. Успешная загрузка всего файла в следующей ячейке дополнительно проверяет выбранную кодировку на всех строках.

`clients.csv` читается в UTF-8, а его столбцы разделены запятыми. `transactions.csv` читается в Windows-1251, столбцы разделены точкой с запятой, а дробная часть суммы отделена запятой, например `1430,95`. У дохода в клиентском файле используется десятичная точка, например `108600.0`. Уже в начале файла операций видны даты ноября и декабря 2023 года, хотя исследуемый период начинается 1 января 2024 года. Эти строки сохраняем и позднее проверяем весь временной диапазон.

In [5]:
clients_raw = pd.read_csv(
    raw_dir / "clients.csv",
    sep=",",
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False
)

transactions_raw = pd.read_csv(
    raw_dir / "transactions.csv",
    sep=";",
    encoding="cp1251",
    dtype="string",
    keep_default_na=False
)

In [6]:
raw_tables = {"clients": clients_raw, "transactions": transactions_raw}
raw_fingerprints = {}
for name, df in raw_tables.items():
    raw_fingerprints[name] = hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()
    print(f"\n{name}: {df.shape[0]} строк, {df.shape[1]} столбцов")
    print("Столбцы:", df.columns.tolist())
    display(df.head(3))
    display(df.sample(3, random_state=42))


clients: 5723 строк, 16 столбцов
Столбцы: ['client_id', 'birth_date', 'gender', 'city', 'segment', 'loyalty_tier', 'card_type', 'registration_date', 'card_open_date', 'income_source', 'monthly_income', 'has_mobile_app', 'app_last_login', 'satisfaction_score', 'branch_code', 'marketing_opt_in']


,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
0,C003407,1964-09-12,М,Москва,Премиальный,Серебро,Премиальная,20.06.2020,20.06.2022,Зарплата в нашем банке,108600.0,Да,2024-12,5,00395,Да
1,C003983,1965-08-21,М,Новосибирск,Массовый,Базовый,Кредитная,23.07.2016,30.07.2016,Не указан,65200.0,Да,2024-01,,00342,Да
2,C000830,1984-02-16,Ж,Новосибирск,Премиальный,Золото,Дебетовая,10.02.2017,30.10.2018,Зарплата в другом банке,167800.0,Да,2024-01,3,00265,Да


,client_id,birth_date,gender,city,segment,loyalty_tier,card_type,registration_date,card_open_date,income_source,monthly_income,has_mobile_app,app_last_login,satisfaction_score,branch_code,marketing_opt_in
1600,C000103,1974-11-05,Ж,Казань,Зарплатный,Базовый,Дебетовая,18.01.2015,04.03.2017,Зарплата в нашем банке,67600.0,Да,2025-06,,00383,Да
1121,C004302,1952-08-26,Ж,Москва,Премиальный,Серебро,Кредитная,15.10.2015,02.03.2019,Зарплата в нашем банке,189100.0,Нет,,3,00605,Да
4329,C000739,1982-03-13,М,г.Екатеринбург,Массовый,Базовый,Кредитная,06.09.2015,01.09.2021,Зарплата в нашем банке,37700.0,Да,2024-06,3,00439,Да



transactions: 616197 строк, 9 столбцов
Столбцы: ['txn_id', 'client_id', 'txn_datetime', 'amount', 'currency', 'category', 'mcc_code', 'channel', 'status']


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
0,990023,C005169,2023-11-23 10:48:18,"1430,95",RUB,АЗС,5541,Карта,успешно
1,990111,C003522,2023-11-25 11:28:29,"311,71",RUB,Продукты,5411,Онлайн-банк,успешно
2,990017,C000223,2023-12-12 20:10:16,"213,46",RUB,Онлайн-сервисы,5815,Карта,успешно


,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
3996,103975,C002318,2024-01-05 18:02:37,"256,59",RUB,Онлайн-сервисы,5815,Токен,успешно
197639,297091,C003435,2024-06-24 09:20:06,"680,76",RUB,Онлайн-сервисы,5815,Карта,успешно
602764,706144,C000918,2025-06-18 12:30:13,"1978,41",RUB,Развлечения и досуг,7832,СБП,успешно


Оба файла загрузили с явно указанными кодировками и разделителями. Параметр `dtype="string"` сохраняет все поля текстом, поэтому код отделения `00395` не теряет ведущие нули, а сумма `1430,95` сохраняет исходную запись. Параметр `keep_default_na=False` оставляет пустые ячейки пустыми строками и не превращает текстовые обозначения вроде `NA` в пропуски автоматически. Это позволяет сначала посмотреть, что именно выдал источник, и только потом принять решения об обработке.

После загрузки проверили размер таблиц, названия столбцов и примеры записей. В `clients_raw` получилось 5723 строки и 16 столбцов, в `transactions_raw` — 616197 строк и 9 столбцов. Для каждой таблицы показали три первые и три случайные строки. `random_state=42` закрепляет случайный выбор, поэтому при повторном запуске будут показаны те же примеры. Для значений строк и индекса каждой исходной таблицы сохранили контрольный отпечаток; он понадобится для проверки неизменности в конце блока.

### 2.3. Паспорт полей

In [7]:
profiles = {}
for name, df in raw_tables.items():
    print(f"\n{name}: {len(df)} строк, {df.shape[1]} столбцов")
    print(f"Память: {df.memory_usage(deep=True).sum():,} байт ({df.memory_usage(deep=True).sum() / 2**20:.2f} МиБ)")
    df.info(memory_usage="deep")
    summary = []
    for col in df.columns:
        s = df[col]
        stripped = s.str.strip()
        valid = s[stripped.ne("")]
        summary.append({
            "Признак": col, "Тип": str(s.dtype),
            "Пустых": int(s.eq("").sum()),
            "Только пробелы": int((s.ne("") & stripped.eq("")).sum()),
            "Пустых, %": round(stripped.eq("").mean() * 100, 2),
            "Пробелы по краям": int(s.ne(stripped).sum()),
            "Уникальных непустых": valid.nunique(),
            "Примеры значений": valid.drop_duplicates().head(3).tolist()
        })
    profiles[name] = pd.DataFrame(summary).set_index("Признак")
    with pd.option_context("display.max_colwidth", None):
        display(profiles[name])


clients: 5723 строк, 16 столбцов
Память: 6,957,726 байт (6.64 МиБ)
<class 'pandas.DataFrame'>
RangeIndex: 5723 entries, 0 to 5722
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   client_id           5723 non-null   string
 1   birth_date          5723 non-null   string
 2   gender              5723 non-null   string
 3   city                5723 non-null   string
 4   segment             5723 non-null   string
 5   loyalty_tier        5723 non-null   string
 6   card_type           5723 non-null   string
 7   registration_date   5723 non-null   string
 8   card_open_date      5723 non-null   string
 9   income_source       5723 non-null   string
 10  monthly_income      5723 non-null   string
 11  has_mobile_app      5723 non-null   string
 12  app_last_login      5723 non-null   string
 13  satisfaction_score  5723 non-null   string
 14  branch_code         5723 non-null   string
 15  marketing_opt_i

,Тип,Пустых,Только пробелы,"Пустых, %",Пробелы по краям,Уникальных непустых,Примеры значений
Признак,,,,,,,
client_id,string,0,0,0.00,0,5668,"[C003407, C003983, C000830]"
birth_date,string,0,0,0.00,0,4850,"[1964-09-12, 1965-08-21, 1984-02-16]"
gender,string,0,0,0.00,0,2,"[М, Ж]"
city,string,0,0,0.00,163,17,"[Москва, Новосибирск, Екатеринбург]"
segment,string,0,0,0.00,0,4,"[Премиальный, Массовый, Зарплатный]"
loyalty_tier,string,0,0,0.00,0,4,"[Серебро, Базовый, Золото]"
card_type,string,0,0,0.00,0,3,"[Премиальная, Кредитная, Дебетовая]"
registration_date,string,0,0,0.00,0,3243,"[20.06.2020, 23.07.2016, 10.02.2017]"
card_open_date,string,0,0,0.00,0,2648,"[20.06.2022, 30.07.2016, 30.10.2018]"



transactions: 616197 строк, 9 столбцов


Память: 390,271,314 байт (372.19 МиБ)


<class 'pandas.DataFrame'>
RangeIndex: 616197 entries, 0 to 616196
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   txn_id        616197 non-null  string
 1   client_id     616197 non-null  string
 2   txn_datetime  616197 non-null  string
 3   amount        616197 non-null  string
 4   currency      616197 non-null  string
 5   category      616197 non-null  string
 6   mcc_code      616197 non-null  string
 7   channel       616197 non-null  string
 8   status        616197 non-null  string
dtypes: string(9)
memory usage: 372.2 MB


,Тип,Пустых,Только пробелы,"Пустых, %",Пробелы по краям,Уникальных непустых,Примеры значений
Признак,,,,,,,
txn_id,string,0,0,0.0,0,614697,"[990023, 990111, 990017]"
client_id,string,0,0,0.0,0,5990,"[C005169, C003522, C000223]"
txn_datetime,string,0,0,0.0,0,560225,"[2023-11-23 10:48:18, 2023-11-25 11:28:29, 2023-12-12 20:10:16]"
amount,string,0,0,0.0,0,322592,"[1430,95, 311,71, 213,46]"
currency,string,0,0,0.0,0,3,"[RUB, EUR, USD]"
category,string,0,0,0.0,0,12,"[АЗС, Продукты, Онлайн-сервисы]"
mcc_code,string,0,0,0.0,0,9,"[5541, 5411, 5815]"
channel,string,0,0,0.0,0,4,"[Карта, Онлайн-банк, Токен]"
status,string,0,0,0.0,0,3,"[успешно, отклонено, возврат]"


В этой ячейке собрали общую информацию о каждом столбце. Сначала показали размер таблицы, объём памяти и типы данных, затем отдельно посчитали пустые строки, строки только из пробелов и значения с пробелами по краям. Для каждой колонки также вывели число разных непустых значений и до трёх примеров. Все проценты пустот рассчитаны от числа строк своей таблицы: для клиентов это 5723, для операций — 616197.

Пробелы удаляются только во временной Series, чтобы проверить заполненность. Исходные значения при этом сохраняются, а число разных значений и примеры считаются по исходному тексту. Поэтому разные написания одного города пока остаются разными значениями. Во всех 25 полях используется тип `string`; оценка памяти составляет 6 957 726 байт для справочника и 390 271 314 байт для операций.

В справочнике пусты 514 значений дохода (8,98%), 2180 месяцев последнего входа (38,09%) и 3431 оценка сервиса (59,95%). В операциях пустых полей не найдено. Строк только из пробелов нет, но у 163 строк города есть пробел по краю. Эти результаты описывают первоначальную заполненность; смысл пустот и специальных значений проверяется ниже, без заполнения или удаления данных.

### 2.4. Фактические значения категориальных полей

In [8]:
category_columns = {
    "clients": ["gender", "city", "segment", "loyalty_tier", "card_type", "income_source",
                "has_mobile_app", "marketing_opt_in", "satisfaction_score"],
    "transactions": ["currency", "category", "mcc_code", "channel", "status"]
}
category_frequencies = {}
for name, cols in category_columns.items():
    df = raw_tables[name]
    frequencies = []
    for col in cols:
        counts = df[col].value_counts(dropna=False)
        table = counts.rename_axis("Значение").reset_index(name="Строк")
        table.insert(0, "Поле", col)
        table["Доля строк, %"] = (table["Строк"] / len(df) * 100).round(2)
        frequencies.append(table)
    category_frequencies[name] = pd.concat(frequencies, ignore_index=True)
    print(f"{name}; знаменатель: {len(df)} строк")
    with pd.option_context("display.max_rows", 100):
        display(category_frequencies[name])

clients; знаменатель: 5723 строк


,Поле,Значение,Строк,"Доля строк, %"
0,gender,М,2882,50.36
1,gender,Ж,2841,49.64
2,city,Москва,1127,19.69
3,city,Санкт-Петербург,664,11.6
4,city,Екатеринбург,629,10.99
5,city,Нижний Новгород,585,10.22
6,city,Новосибирск,553,9.66
7,city,Казань,543,9.49
8,city,г. Москва,190,3.32
9,city,казань,171,2.99


transactions; знаменатель: 616197 строк


,Поле,Значение,Строк,"Доля строк, %"
0,currency,RUB,613688,99.59
1,currency,EUR,1275,0.21
2,currency,USD,1234,0.2
3,category,Продукты,162273,26.33
4,category,Транспорт,64675,10.5
5,category,Одежда и обувь,63380,10.29
6,category,АЗС,56733,9.21
7,category,Развлечения и досуг,53540,8.69
8,category,Кафе и рестораны,47585,7.72
9,category,Аптеки,44761,7.26


Здесь посчитали частоты всех значений в выбранных категориальных полях. Для каждого значения таблица показывает число исходных строк и их долю от общего размера соответствующего источника. В клиентской таблице знаменатель равен 5723 строкам, в операциях — 616197. Поскольку повторы ещё не обработаны, это частоты записей выгрузки; число уникальных клиентов или подтверждённо разных операций проверяется отдельно.

Такой просмотр позволяет увидеть фактический набор названий и сравнить его с описанием. В частности, у города показано 17 написаний, у категории траты — 12, а в валюте встречаются RUB, EUR и USD. В `income_source` есть явное значение `Не указан`, а в оценке сервиса отдельной строкой видны пустые значения. Частоты позволяют заметить эти особенности, но сами по себе не устанавливают, какие названия следует объединить и какие пустоты заполнять. Ниже проверяем написания, смысл отдельных значений и согласованность полей.

#### Варианты написания в категориальных полях

In [9]:
category_review = []
for name, cols in category_columns.items():
    for col in cols:
        s = raw_tables[name][col]
        category_review.append({
            "Таблица": name, "Поле": col, "Исходных значений": s.nunique(),
            "После пробелов и регистра": s.str.strip().str.casefold().nunique(),
            "Строк с краевыми пробелами": int(s.ne(s.str.strip()).sum())
        })
display(pd.DataFrame(category_review))

,Таблица,Поле,Исходных значений,После пробелов и регистра,Строк с краевыми пробелами
0,clients,gender,2,2,0
1,clients,city,17,13,163
2,clients,segment,4,4,0
3,clients,loyalty_tier,4,4,0
4,clients,card_type,3,3,0
5,clients,income_source,3,3,0
6,clients,has_mobile_app,2,2,0
7,clients,marketing_opt_in,2,2,0
8,clients,satisfaction_score,6,6,0
9,transactions,currency,3,3,0


Здесь для каждого категориального поля сравнили число разных исходных значений с числом значений после временного удаления краевых пробелов и приведения регистра. Если количество уменьшается, значит, часть названий отличается только написанием. Само изменение количества ещё не даёт правила очистки, поскольку сокращения и похожие слова нужно проверять по смыслу.

У города количество значений уменьшается с 17 до 13, у категории операции — с 12 до 10. Для остальных проверенных полей количество остаётся прежним, и краевых пробелов в них нет. В ранее выведенных частотах не обнаружены дополнительные значения сегмента, уровня лояльности, типа карты, бинарных признаков, оценки, валюты, канала и статуса, которые требовали бы объединения написаний. Отдельные сокращения городов и вариант `Кафе/рестораны` этой проверкой полностью не объединяются, поэтому для них ниже рассматриваются явные словари.

In [10]:
city_aliases = {
    "г. Москва": "Москва", "МОСКВА": "Москва", " москва": "Москва",
    "г. Санкт-Петербург": "Санкт-Петербург", "Санкт Петербург": "Санкт-Петербург",
    "СПб": "Санкт-Петербург", "г.Екатеринбург": "Екатеринбург",
    "ЕКАТЕРИНБУРГ": "Екатеринбург", "Н. Новгород": "Нижний Новгород",
    "казань": "Казань", "г. Новосибирск": "Новосибирск"
}
city_check = clients_raw["city"].replace(city_aliases)
city_review = clients_raw["city"].value_counts().rename_axis("Исходное").reset_index(name="Строк")
city_review["Единое название"] = city_review["Исходное"].replace(city_aliases)
city_review["Исходное"] = city_review["Исходное"].map(repr)
display(city_review)
print("Строк с изменяемым написанием:", city_check.ne(clients_raw["city"]).sum())
print("Городов после применения словаря во временной Series:", city_check.nunique())

,Исходное,Строк,Единое название
0,'Москва',1127,Москва
1,'Санкт-Петербург',664,Санкт-Петербург
2,'Екатеринбург',629,Екатеринбург
3,'Нижний Новгород',585,Нижний Новгород
4,'Новосибирск',553,Новосибирск
5,'Казань',543,Казань
6,'г. Москва',190,Москва
7,'казань',171,Казань
8,'МОСКВА',166,Москва
9,' москва',163,Москва


Строк с изменяемым написанием: 1622
Городов после применения словаря во временной Series: 6


В этой ячейке перечислили конкретные варианты городов и рядом показали название, к которому их можно привести. Исходные значения выведены в кавычках через `repr`, чтобы пробел в начале ` москва` был виден. Сокращения `СПб` и `Н. Новгород` соответствуют Санкт-Петербургу и Нижнему Новгороду; другие варианты отличаются регистром, дефисом или префиксом `г.`.

По этому словарю 17 исходных написаний сводятся к 6 городам. Текст изменился бы у 1622 из 5723 строк, в том числе у 163 строк со значением ` москва`. Пока словарь применён только к отдельной проверочной Series, исходный `clients_raw.city` не менялся. При очистке можно использовать именно этот перечень замен, а новые написания проверять отдельно. Разные города у одного и того же клиента такой словарь не исправляет; эти случаи рассматриваются в проверке повторов.

#### Категории операций и MCC

In [11]:
category_aliases = {
    "Связь и Интернет": "Связь и интернет",
    "кафе и рестораны": "Кафе и рестораны",
    "Кафе/рестораны": "Кафе и рестораны"
}
category_check = transactions_raw["category"].replace(category_aliases)
category_mcc_raw = pd.crosstab(transactions_raw["category"], transactions_raw["mcc_code"])
category_variants = pd.DataFrame({
    "Строк": category_mcc_raw.sum(axis=1),
    "Единое написание": category_mcc_raw.index.to_series().replace(category_aliases),
    "Основной MCC": category_mcc_raw.idxmax(axis=1),
    "С другим MCC": category_mcc_raw.sum(axis=1) - category_mcc_raw.max(axis=1)
})
display(category_variants)
print("Строк с изменяемым написанием:", category_check.ne(transactions_raw["category"]).sum())
print("Категорий после словаря:", category_check.nunique())

,Строк,Единое написание,Основной MCC,С другим MCC
category,,,,
АЗС,56733,АЗС,5541,145
Аптеки,44761,Аптеки,5912,121
Кафе и рестораны,47585,Кафе и рестораны,5812,129
Кафе/рестораны,15751,Кафе и рестораны,5812,31
Одежда и обувь,63380,Одежда и обувь,5651,176
Онлайн-сервисы,43087,Онлайн-сервисы,5815,111
Продукты,162273,Продукты,5411,393
Развлечения и досуг,53540,Развлечения и досуг,7832,134
Связь и Интернет,19354,Связь и интернет,4814,46


Строк с изменяемым написанием: 51025


Категорий после словаря: 9


В этой ячейке проверили три дополнительных написания категорий и для каждого исходного названия показали наиболее частый MCC. Временный словарь объединяет `Связь и Интернет` со `Связь и интернет`, а `кафе и рестораны` и `Кафе/рестораны` — с `Кафе и рестораны`. При этом таблица сохраняет частоты каждого исходного написания и показывает, сколько его строк имеют другой MCC.

Для смены написания требуется 51025 замен текста: 19354 строки связи, 15920 строк кафе в нижнем регистре и 15751 строка с названием через косую черту. После словаря остаётся 9 категорий, как в описании источника. У обоих написаний связи основной MCC равен `4814`, у всех трёх написаний кафе — `5812`, что поддерживает предложенное объединение названий. Однако у части строк MCC отличается от основного, поэтому одной нормализацией написаний нельзя подтвердить правильность каждой конкретной пары. Эта проверка выполняется в следующей ячейке.

In [12]:
category_mcc_check = pd.crosstab(transactions_raw["mcc_code"], category_check)
display(category_mcc_check)
main_category_by_mcc = category_mcc_check.idxmax(axis=1)
mcc_disagreement = category_check.ne(transactions_raw["mcc_code"].map(main_category_by_mcc))
print("Строк с несовпадающей парой:", mcc_disagreement.sum())
print("Уникальных txn_id в этих строках:",
      transactions_raw.loc[mcc_disagreement, "txn_id"].nunique())
display(transactions_raw.loc[mcc_disagreement, ["txn_id", "category", "mcc_code"]].head(5))

category,АЗС,Аптеки,Кафе и рестораны,Одежда и обувь,Онлайн-сервисы,Продукты,Развлечения и досуг,Связь и интернет,Транспорт
mcc_code,,,,,,,,,
4111,17,16,22,24,17,47,18,12,64513
4814,24,12,31,21,15,52,17,48362,12
5411,18,16,26,22,9,161880,13,19,19
5541,56588,8,25,25,13,51,19,27,19
5651,22,14,22,63204,15,50,18,18,18
5812,21,17,79059,21,15,47,15,12,22
5815,13,18,30,18,42976,59,19,11,26
5912,18,44640,21,19,11,41,15,18,24
7832,12,20,20,26,16,46,53406,13,22


Строк с несовпадающей парой: 1569
Уникальных txn_id в этих строках: 1562


,txn_id,category,mcc_code
214,100210,Одежда и обувь,5411
412,100408,Связь и Интернет,7832
866,100859,Продукты,4814
983,100976,Продукты,5812
1185,101178,Онлайн-сервисы,5651


После временного объединения написаний проверили, как категории связаны с MCC. В таблице строки соответствуют кодам MCC, столбцы — категориям, а число в ячейке показывает, сколько раз встретилась такая пара. Для каждого MCC нашли наиболее частую категорию и сравнили с ней каждую исходную строку операции. Это внутреннее сравнение по самой выгрузке; наиболее частая пара не является независимым подтверждением правильной категории.

Подтверждено, что 1569 строк с 1562 уникальными `txn_id` отличаются от преобладающих пар. Например, у операции `100210` указаны `Одежда и обувь` и MCC `5411`, хотя в этой выгрузке код `5411` преимущественно связан с продуктами. Ошибка категории, ошибка MCC и особенности классификации источника остаются возможными объяснениями. Определить правильное значение конкретной строки по этим двум полям нельзя, поэтому обнаруженные пары нельзя автоматически исправлять по большинству.

На следующем этапе можно сохранить оба исходных значения и добавить отдельную отметку о несовпадении. Если сверить значения с источником не получится, нужно сравнить выводы по категориям при сохранении исходных категорий и при отдельном учёте этих строк вне обычных категорий. Сравниваются размеры категорий и их месячная динамика; стратегия выбирается после такой проверки и документируется как допущение. Удалять эти операции из всех метрик активности только из-за неуточнённой категории оснований нет.

In [13]:
numeric_values = {}
numeric_rows = []
for name, col in [("clients", "monthly_income"), ("transactions", "amount")]:
    text = raw_tables[name][col].str.strip()
    numeric = pd.to_numeric(text.str.replace(",", ".", regex=False), errors="coerce")
    numeric_values[col] = numeric
    numeric_rows.append({
        "Поле": name + "." + col, "Пустых": int(text.eq("").sum()),
        "Нечисловых непустых": int((text.ne("") & numeric.isna()).sum()),
        "Бесконечных": int((numeric.notna() & ~np.isfinite(numeric)).sum()),
        "Нулей": int(numeric.eq(0).sum()), "Отрицательных": int(numeric.lt(0).sum()),
        "Минимум": numeric.min(), "Максимум": numeric.max()
    })
display(pd.DataFrame(numeric_rows).set_index("Поле"))
negative_income = numeric_values["monthly_income"].lt(0)
display(clients_raw.loc[negative_income, "monthly_income"].value_counts().to_frame("Строк"))
print("Уникальных клиентов с отрицательным доходом:",
      clients_raw.loc[negative_income, "client_id"].nunique())

,Пустых,Нечисловых непустых,Бесконечных,Нулей,Отрицательных,Минимум,Максимум
Поле,,,,,,,
clients.monthly_income,514,0,0,0,57,-999.00,867100.00
transactions.amount,0,0,0,0,7404,-47670.83,114061.89


,Строк
monthly_income,
-999.0,57


Уникальных клиентов с отрицательным доходом: 56


Здесь проверили, можно ли прочитать доход и сумму операции как числа. Для этого создали временные числовые Series, предварительно заменив десятичную запятую точкой только в проверяемом тексте. Затем отдельно посчитали пустые значения, непустые строки, которые не удалось прочитать, бесконечности, нули и отрицательные числа. Успешное чтение числа проверяет его формат, поэтому дополнительно нужно смотреть, имеет ли это число понятный смысл.

Нечисловых непустых значений, бесконечностей и нулей не найдено. В доходе 514 пустых строк и 57 отрицательных значений; все отрицательные значения равны `-999.0` и относятся к 56 уникальным клиентам. Такое повторяющееся число похоже на специальный код отсутствия, но в описании источника его значение не раскрыто. До уточнения его нельзя использовать как обычный доход или заменять нулём. В суммах операций отрицательных значений 7404; следующая проверка показывает, с какими статусами они связаны.

In [14]:
amount_check = numeric_values["amount"]
amount_sign = pd.Series("положительная", index=transactions_raw.index)
amount_sign.loc[amount_check.lt(0)] = "отрицательная"
amount_sign.loc[amount_check.eq(0)] = "нулевая"
display(pd.crosstab(transactions_raw["status"], amount_sign).rename_axis(columns="Знак суммы"))
display(pd.DataFrame({"Валюта": transactions_raw["currency"], "Сумма": amount_check})
        .groupby("Валюта").agg(Строк=("Сумма", "size"), Минимум=("Сумма", "min"), Максимум=("Сумма", "max")))

Знак суммы,отрицательная,положительная
status,,
возврат,7404,0
отклонено,0,12114
успешно,0,596679


,Строк,Минимум,Максимум
Валюта,,,
EUR,1275,-59.15,314.2
RUB,613688,-47670.83,114061.89
USD,1234,-68.07,234.76


В первой таблице сопоставили знак суммы со статусом операции. Это нужно, чтобы не признать отрицательную сумму ошибкой только из-за её знака. Все 7404 отрицательные суммы относятся к возвратам. У 596679 успешных и 12114 отклонённых операций суммы положительные; нулевых сумм нет. Получается, что отрицательные суммы согласуются со статусом возврата, а положительная сумма отклонённой операции сама по себе не означает, что деньги были списаны.

Во второй таблице показали количество строк и диапазон сумм отдельно для каждой валюты. В RUB находится 613688 строк, в EUR — 1275, в USD — 1234. Эти суммы нельзя складывать в один денежный показатель без правила пересчёта, поскольку они выражены в разных единицах. При дальнейшей подготовке нужно отдельно описать учёт успешных оплат, отказов и возвратов. Проверка распределений и выбросов здесь не проводится, а исходные суммы и статусы остаются без изменений.

#### Пустоты и специальные значения

In [15]:
empty_review = pd.concat({name: profile[["Пустых", "Только пробелы"]]
                          for name, profile in profiles.items()}, names=["Таблица"])
display(empty_review.loc[empty_review.sum(axis=1).gt(0)])
print("Пустых ячеек:", empty_review["Пустых"].sum())
display(clients_raw["income_source"].value_counts(dropna=False).to_frame("Строк"))

income_frequency = (clients_raw["monthly_income"].value_counts(dropna=False)
                    .rename_axis("Исходное значение").reset_index(name="Строк"))
income_frequency["Число"] = pd.to_numeric(income_frequency["Исходное значение"], errors="coerce")
display(income_frequency.sort_values("Число").head(6))

income_state = pd.Series("положительное число", index=clients_raw.index)
income_state.loc[clients_raw["monthly_income"].str.strip().eq("")] = "пустая строка"
income_state.loc[numeric_values["monthly_income"].lt(0)] = "отрицательное число"
display(pd.crosstab(clients_raw["income_source"], income_state).rename_axis(columns="Запись дохода"))

review_income_limit = 1000
small_income = numeric_values["monthly_income"].gt(0) & numeric_values["monthly_income"].lt(review_income_limit)
display(pd.Series({
    "Строк с положительным доходом меньше 1000": int(small_income.sum()),
    "Уникальных клиентов": clients_raw.loc[small_income, "client_id"].nunique(),
    "Минимум в этом диапазоне, руб.": numeric_values["monthly_income"][small_income].min(),
    "Максимум в этом диапазоне, руб.": numeric_values["monthly_income"][small_income].max(),
    "Минимум остальных положительных, руб.": numeric_values["monthly_income"].loc[
        numeric_values["monthly_income"].ge(review_income_limit)].min()
}, name="Значение").to_frame())

Пустых  Только пробелы
Таблица Признак                                   
clients monthly_income         514               0
        app_last_login        2180               0
        satisfaction_score    3431               0

Пустых ячеек: 6125


,Строк
income_source,
Зарплата в нашем банке,2678
Зарплата в другом банке,1569
Не указан,1476


,Исходное значение,Строк,Число
1,-999.0,57,-999.0
1089,26.5,1,26.5
1605,28.9,1,28.9
1553,31.0,1,31.0
1489,32.2,1,32.2
1292,32.5,1,32.5


Запись дохода,отрицательное число,положительное число,пустая строка
income_source,,,
Зарплата в другом банке,19,1503,47
Зарплата в нашем банке,27,2554,97
Не указан,11,1095,370


,Значение
Строк с положительным доходом меньше 1000,184.0
Уникальных клиентов,182.0
"Минимум в этом диапазоне, руб.",26.5
"Максимум в этом диапазоне, руб.",548.6
"Минимум остальных положительных, руб.",17300.0


В этой проверке исходили из значений, которые действительно встретились в файле. Полные частоты категориальных полей показаны выше, а для чисел и дат уже проверено, какие строки читаются в ожидаемом формате. Здесь отдельно собрали пустые ячейки, вывели все три значения источника зарплаты и посмотрели наименьшие записанные доходы с числом повторений. Отбора по заранее заданным словам, которые нужно считать пропусками, нет. Таблицы используются для просмотра и обсуждения смысла значений; исходные столбцы не изменяются.

В трёх полях справочника найдено 6125 пустых ячеек: 514 доходов, 2180 месяцев входа и 3431 оценка. Один человек может иметь несколько пустых полей, поэтому это число не равно числу клиентов. Среди фактических категорий источника зарплаты находится `Не указан`, 1476 строк. По смыслу названия источник зарплаты, вероятно, не уточнён, но правила ввода этой категории не описаны. При этом у 1095 таких строк доход положительный, у 370 доход пустой, а у 11 записано отрицательное число. Поэтому значение `Не указан` нельзя автоматически распространять на соседнее поле дохода, трактовать как отсутствие зарплаты или заменять пропуском без решения о смысле конкретного столбца.

Все 57 отрицательных доходов равны `-999.0`. Точное повторение одного отрицательного числа в поле месячного дохода делает его подозрительным обозначением, однако назначение этого числа в источнике неизвестно. Оно остаётся числом в проверочной Series и не превращается в пропуск автоматически. Для дальнейшей обработки нужно уточнить, является ли это служебным кодом; до уточнения рассматриваем варианты обработки как предположения и проверяем их влияние на выводы по доходу.

Просмотр наименьших значений также показал положительные суммы порядка десятков и сотен при заявленной единице «рубли». Для количественного описания отдельно посчитали диапазон от 0 до 1000 рублей: в нём 184 строки с 182 уникальными клиентами, значения от 26,5 до 548,6. Следующее наименьшее положительное значение вне этого диапазона равно 17300. Граница 1000 выбрана только для описания замеченного разрыва, она не является порогом ошибки или правилом очистки. Различие масштаба может быть связано с рублями и тысячами рублей, но подтверждения этому нет; малые суммы также не объявляются пропусками и не умножаются автоматически.

In [16]:
login_empty = clients_raw["app_last_login"].eq("")
display(pd.crosstab(clients_raw["has_mobile_app"],
                    login_empty.map({True: "пусто", False: "указан"}))
        .rename_axis(columns="Месяц входа"))

Месяц входа,пусто,указан
has_mobile_app,,
Да,0,3543
Нет,2180,0


В этой таблице проверили, у каких клиентов отсутствует месяц последнего входа в приложение. Строки показывают значение `has_mobile_app`, а столбцы — заполнен ли `app_last_login`. Такая проверка нужна, чтобы отличить возможную потерю даты от случая, когда поле связано с отсутствием приложения в текущем снимке.

Все 2180 пустых месяцев входа находятся у строк с `has_mobile_app = Нет`. У всех 3543 строк с `Да` месяц входа указан, поэтому несогласованных сочетаний этой пары полей не найдено. Дату входа для клиентов без приложения заполнять вымышленным месяцем не нужно. При этом снимок описывает текущее состояние, поэтому он не доказывает, что эти клиенты никогда не устанавливали приложение. Историю установок и удаления приложения по данным полям восстановить нельзя.

### 2.5. Ключи и связь таблиц

In [17]:
key_rows = []
for name, col in [("clients", "client_id"), ("transactions", "txn_id"), ("transactions", "client_id")]:
    s = raw_tables[name][col]
    stripped = s.str.strip()
    key_rows.append({
        "Таблица": name, "Ключ": col, "Строк": len(s),
        "Пустых после strip": int(stripped.eq("").sum()),
        "С пробелами по краям": int(s.ne(stripped).sum()),
        "Уникальных исходных": s[stripped.ne("")].nunique(),
        "Уникальных после strip": stripped[stripped.ne("")].nunique()
    })
key_summary = pd.DataFrame(key_rows)
display(key_summary)

,Таблица,Ключ,Строк,Пустых после strip,С пробелами по краям,Уникальных исходных,Уникальных после strip
0,clients,client_id,5723,0,0,5668,5668
1,transactions,txn_id,616197,0,0,614697,614697
2,transactions,client_id,616197,0,0,5990,5990


В этой ячейке проверили заполненность и написание идентификаторов. Для `client_id` в обеих таблицах и `txn_id` в операциях посчитали количество строк, пустых ключей, значений с краевыми пробелами и разных ID. Число разных ID также посчитали после временного удаления пробелов, чтобы выяснить, не представлены ли одинаковые ключи несколькими написаниями.

Пустых ключей и пробелов по краям не найдено, количество разных ID после этой проверки не изменилось. В справочнике 5668 уникальных клиентов, в операциях — 5990 клиентских ID и 614697 ID операций. Повторение `client_id` в операциях ожидаемо, поскольку один человек может совершать много платежей. Для справочника клиентов и идентификатора операции нужно отдельно проверить повторяющиеся строки: одинаковый ID ещё не показывает, полностью ли совпадают записи.

In [18]:
for name, key in [("clients", "client_id"), ("transactions", "txn_id")]:
    df = raw_tables[name]

    repeated = df.loc[df[key].duplicated(keep=False), key].nunique()
    exact = df.duplicated().sum()
    conflicting = df.drop_duplicates().groupby(key).size().gt(1).sum()

    print(f"\n{name}:")
    print(f"ID с повторами: {repeated}")
    print("ID ровно с двумя строками:", df[key].value_counts().eq(2).sum())
    print("Все строки в полных дублях:", df.duplicated(keep=False).sum())
    print(f"Полностью одинаковых повторных строк: {exact}")
    print(f"ID с различающимися записями: {conflicting}")


clients:
ID с повторами: 55
ID ровно с двумя строками: 55
Все строки в полных дублях: 82
Полностью одинаковых повторных строк: 41
ID с различающимися записями: 14



transactions:
ID с повторами: 1500


ID ровно с двумя строками: 1500


Все строки в полных дублях: 2612
Полностью одинаковых повторных строк: 1306
ID с различающимися записями: 194


Здесь разделили два случая повторения данных. Сначала посчитали ID, которые встречаются несколько раз, и проверили, что каждый такой ID представлен ровно двумя строками. Затем нашли полностью одинаковые копии и отдельно посчитали ID, у которых после временного исключения полных копий остаются различающиеся записи. Временная таблица используется только для сравнения, исходные строки не удаляются.

В справочнике повторяются 55 клиентских ID. У 41 из них строки полностью одинаковы: это 82 строки, из которых 41 является дополнительной копией. У остальных 14 клиентов значения между строками различаются. В операциях повторяются 1500 `txn_id`: у 1306 ID строки полностью одинаковы, ещё у 194 есть различия. Полные копии и разные записи требуют отдельных правил, поэтому ниже проверяем, какие именно поля не совпадают.

In [19]:
# Убираем точные копии строк только во временной таблице
unique_rows = transactions_raw.drop_duplicates()

# Находим ID, для которых остались разные записи
conflicts = unique_rows[
    unique_rows.duplicated("txn_id", keep=False)
]

# Показываем примеры конфликтов
display(conflicts.sort_values("txn_id").head(10))

# Считаем, у скольких ID различается каждый столбец
differences = conflicts.groupby("txn_id").nunique().gt(1).sum()
display(differences[differences > 0])

,txn_id,client_id,txn_datetime,amount,currency,category,mcc_code,channel,status
789,100784,C005323,2024-01-02,"756,84",RUB,Связь и интернет,4814,СБП,успешно
790,100784,C005323,2024-01-02 09:14:28,"756,84",RUB,Связь и интернет,4814,СБП,успешно
1290,101283,C003426,2024-01-02,"899,8",RUB,Продукты,5411,Онлайн-банк,успешно
1291,101283,C003426,2024-01-02 17:43:27,"899,8",RUB,Продукты,5411,Онлайн-банк,успешно
3567,103548,C000203,2024-01-05,"8662,76",RUB,Одежда и обувь,5651,СБП,успешно
3568,103548,C000203,2024-01-05 14:36:05,"8662,76",RUB,Одежда и обувь,5651,СБП,успешно
7810,107784,C000816,2024-01-09 12:16:41,"2170,2",RUB,Продукты,5411,Карта,успешно
7811,107784,C000816,2024-01-09,"2170,2",RUB,Продукты,5411,Карта,успешно
8622,108591,C001647,2024-01-10 13:26:36,"381,65",RUB,Связь и Интернет,4814,Карта,успешно
8623,108591,C001647,2024-01-10,"381,65",RUB,Связь и Интернет,4814,Карта,успешно


txn_datetime    194
dtype: int64

Для этой проверки создали временную таблицу операций без полностью одинаковых копий и оставили только ID, которым соответствуют разные строки. Первые десять строк показаны как примеры, чтобы было видно, как выглядят пары. После просмотра примеров отдельно посчитали число разных значений каждого поля внутри каждого `txn_id`, поэтому результат проверки относится ко всем 194 ID.

У всех 194 пар различается только `txn_datetime`. Клиент, сумма, валюта, категория, MCC, канал и статус совпадают. В примерах одна строка содержит только дату, другая — дату и время. Пока это показывает разную полноту записи времени, но для выбора более полной строки нужно ещё проверить совпадение календарного дня во всех парах. Эта проверка выполняется в следующей ячейке.

In [20]:
check = conflicts.assign(
    day=pd.to_datetime(conflicts["txn_datetime"].str[:10], format="%Y-%m-%d", errors="coerce"),
    has_time=conflicts["txn_datetime"].str.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"),
    date_only=conflicts["txn_datetime"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
)
summary = check.groupby("txn_id").agg(
    rows=("txn_id", "size"), days=("day", "nunique"),
    with_time=("has_time", "sum"), without_time=("date_only", "sum")
)
print("Всего конфликтующих ID:", len(summary))
print("Нераспознанных календарных дней:", check["day"].isna().sum())
print("ID с разными календарными днями:", summary["days"].ne(1).sum())
print("ID, у которых не две записи:", summary["rows"].ne(2).sum())
print("ID без ровно одной записи с временем:", summary["with_time"].ne(1).sum())
print("ID без ровно одной записи только с датой:", summary["without_time"].ne(1).sum())

Всего конфликтующих ID: 194
Нераспознанных календарных дней: 0
ID с разными календарными днями: 0
ID, у которых не две записи: 0
ID без ровно одной записи с временем: 0
ID без ровно одной записи только с датой: 0


В каждой из 194 пар проверили количество строк, календарный день и наличие записи каждого формата. День прочитали как дату, чтобы не ограничиваться сравнением похожего текста. Отдельно проверили, что в каждой паре есть ровно одна строка с временем и ровно одна строка только с датой.

Нераспознанных дней и исключений из этих условий нет. Во всех 194 случаях представлены две строки одной операции с одинаковым календарным днём, причём остальные поля, как показала предыдущая проверка, совпадают. Поэтому для следующего этапа есть обоснованное правило: оставить запись с указанным временем и убрать вторую строку только с датой. Так будут устранены 194 повторные строки без потери имеющихся сведений. Сейчас правило только проверено, к исходной таблице оно не применяется.

#### Чем различаются записи клиентов

In [21]:
client_unique_rows = clients_raw.drop_duplicates()
client_conflicts = client_unique_rows.loc[client_unique_rows.duplicated("client_id", keep=False)]
client_field_differences = client_conflicts.groupby("client_id").nunique().gt(1)
display(client_field_differences.sum().loc[client_field_differences.sum().gt(0)].to_frame("Клиентов"))
with pd.option_context("display.max_colwidth", None):
    display(client_conflicts.groupby("client_id")[["city", "segment"]].agg(list))
print("Различающихся строк клиентов:", len(client_conflicts))
print("Клиентов с разными городами после словаря:",
      city_check.loc[client_conflicts.index].groupby(client_conflicts["client_id"]).nunique().gt(1).sum())

,Клиентов
city,14
segment,9


,city,segment
client_id,,
C000327,"[Санкт-Петербург, Новосибирск]","[Массовый, Массовый]"
C000470,"[Москва, Екатеринбург]","[Премиальный, Массовый]"
C000983,"[Москва, Екатеринбург]","[Массовый, Массовый]"
C001033,"[Екатеринбург, Казань]","[Массовый, Массовый]"
C001105,"[Санкт-Петербург, Казань]","[Массовый, Премиальный]"
C002218,"[Санкт-Петербург, Екатеринбург]","[Зарплатный, Зарплатный]"
C002322,"[Москва, Екатеринбург]","[Зарплатный, Массовый]"
C003637,"[Нижний Новгород, г. Москва]","[Премиальный, Зарплатный]"
C004239,"[ москва, Санкт-Петербург]","[Зарплатный, Private]"


Различающихся строк клиентов: 28
Клиентов с разными городами после словаря: 14


В клиентской таблице отделили полные копии только во временном наборе, после чего сравнили поля внутри каждого повторяющегося `client_id`. Первая таблица показывает, у скольких клиентов различается каждое поле. Во второй приведены сами пары значений города и сегмента, чтобы характер различий можно было проверить по данным.

Подтверждены 28 различающихся строк у 14 клиентов: у всех 14 различается город, у 9 из них дополнительно различается сегмент; остальные поля совпадают. После применения проверочного словаря написаний города все 14 расхождений сохраняются. Например, для `C000470` указаны Москва и Екатеринбург, а также массовый и премиальный сегменты. Это разные значения клиентских характеристик, которые невозможно объяснить одним регистром или сокращением. Возможные причины — разные версии сведений или ошибка при записи и объединении источников, но ни одна причина по этому файлу не подтверждена.

В справочнике нет даты обновления строки, поэтому выбрать актуальный город и сегмент по имеющимся полям нельзя. На следующем этапе сначала нужно попытаться уточнить эти сведения у источника. Если уточнения не будет, один из возможных способов — сохранить совпадающие поля клиента, а спорные город и сегмент учитывать как неустановленные, не выбирая первую или последнюю строку. Это предлагаемая стратегия, а не восстановленное правильное значение. Перед её принятием нужно сравнить размеры, доли и состав групп по городу и сегменту при отдельной группе неустановленных значений и при исключении спорных клиентов только из соответствующего разреза. Общую клиентскую активность следует проверить с сохранением этих клиентов, поскольку остальные поля у них согласованы.

In [22]:
client_ids = clients_raw["client_id"].drop_duplicates()
txn_client_ids = transactions_raw["client_id"].drop_duplicates()
unknown_client = ~transactions_raw["client_id"].isin(client_ids)
relation_summary = pd.Series({
    "Уникальных ID в справочнике": len(client_ids),
    "Уникальных ID в операциях": len(txn_client_ids),
    "Уникальных ID в обеих таблицах": int(client_ids.isin(txn_client_ids).sum()),
    "Уникальных ID справочника без операций": int((~client_ids.isin(txn_client_ids)).sum()),
    "Строк операций без ID в справочнике": int(unknown_client.sum()),
    "Уникальных ID операций вне справочника": transactions_raw.loc[unknown_client, "client_id"].nunique(),
    "Максимум строк операций на один ID": int(transactions_raw["client_id"].value_counts().max())
}, name="Количество")
display(relation_summary.to_frame())

,Количество
Уникальных ID в справочнике,5668
Уникальных ID в операциях,5990
Уникальных ID в обеих таблицах,5658
Уникальных ID справочника без операций,10
Строк операций без ID в справочнике,28149
Уникальных ID операций вне справочника,332
Максимум строк операций на один ID,833


Здесь проверили, какие уникальные клиенты встречаются в обеих таблицах. Сначала получили отдельные перечни `client_id`, затем через `isin` проверили принадлежность каждого ID другому перечню. Полного объединения таблиц нет, поэтому повторные записи в справочнике не размножают строки операций. Число клиентов считается по уникальным ID, а число операций вне справочника — по исходным строкам операций.

В обеих таблицах присутствуют 5658 уникальных клиентов. У 10 клиентов справочника нет ни одной строки операции во всей выгрузке. Ещё 332 клиента присутствуют в операциях, но отсутствуют в справочнике; на них приходится 28149 строк. Их идентификаторы заполнены, однако соответствующей клиентской записи в `clients` нет. Максимум строк операций на один ID составляет 833, пока с учётом повторов.

По заданию справочник содержит действующих клиентов на 30.06.2025, поэтому различие охвата может быть связано с датой и составом снимка. По этим данным нельзя установить, закрыты ли отсутствующие клиенты или не попали в справочник по другой причине. После распознавания всех дат ниже проверяем, сколько их операций относится именно к исследуемому интервалу. Отсутствие операций у десяти клиентов также пока не даёт ответа, наблюдается ли неактивность или неполнота выгрузки.

### 2.6. Даты и временное покрытие

In [23]:
date_formats = [
    ("clients", "birth_date", "%Y-%m-%d"),
    ("clients", "registration_date", "%d.%m.%Y"),
    ("clients", "card_open_date", "%d.%m.%Y"),
    ("clients", "app_last_login", "%Y-%m"),
    ("transactions", "txn_datetime", "%Y-%m-%d %H:%M:%S")
]
parsed_dates = {}
date_rows = []
for name, col, fmt in date_formats:
    s = raw_tables[name][col]
    parsed = pd.to_datetime(s, format=fmt, errors="coerce")
    parsed_dates[col] = parsed
    nonempty = s.str.strip().ne("")
    invalid = nonempty & parsed.isna()
    date_rows.append({
        "Поле": col, "Формат": fmt, "Непустых": int(nonempty.sum()),
        "Не распознано непустых": int(invalid.sum()),
        "Минимум распознанных": str(parsed.min()), "Максимум распознанных": str(parsed.max()),
        "Примеры нераспознанных": s[invalid].drop_duplicates().head(3).tolist()
    })
date_summary = pd.DataFrame(date_rows).set_index("Поле")
display(date_summary)

,Формат,Непустых,Не распознано непустых,Минимум распознанных,Максимум распознанных,Примеры нераспознанных
Поле,,,,,,
birth_date,%Y-%m-%d,5723,0,1936-07-24 00:00:00,2002-09-16 00:00:00,[]
registration_date,%d.%m.%Y,5723,0,2010-05-18 00:00:00,2025-03-13 00:00:00,[]
card_open_date,%d.%m.%Y,5723,0,2016-01-01 00:00:00,2025-05-16 00:00:00,[]
app_last_login,%Y-%m,3543,0,2024-01-01 00:00:00,2025-06-01 00:00:00,[]
txn_datetime,%Y-%m-%d %H:%M:%S,616197,49237,2023-11-23 10:48:18,2026-07-26 11:47:38,"[2024-01-01, 2024-01-02, 2024-01-03]"


В этой ячейке каждое поле даты прочитали с ожидаемым форматом и сохранили результат отдельно от исходной таблицы. Для рождения используется `YYYY-MM-DD`, для регистрации и открытия карты — `DD.MM.YYYY`, для последнего входа — `YYYY-MM`. У операций сначала проверяется полный формат с датой и временем. Параметр `errors="coerce"` позволяет получить пустой результат там, где строка не прочиталась, и затем посчитать такие случаи, не останавливая всё исследование.

При подсчёте ошибок отдельно исключили исходные пустые строки, чтобы отсутствие месяца входа не смешивалось с неверно записанной датой. Все заполненные клиентские даты распознаны: по 5723 значения в полях рождения, регистрации и открытия карты, а также 3543 месяца входа. В операциях полный формат не прочитал 49237 значений. В примерах видно, что они записаны только как календарная дата без времени. Следующая ячейка проверяет этот второй формат на всех соответствующих строках.

In [24]:
txn_text = transactions_raw["txn_datetime"]
date_only = txn_text.str.fullmatch(r"\d{4}-\d{2}-\d{2}")
txn_date_only = pd.to_datetime(txn_text.where(date_only), format="%Y-%m-%d", errors="coerce")
txn_calendar = parsed_dates["txn_datetime"].fillna(txn_date_only)
format_summary = pd.Series({
    "Распознано YYYY-MM-DD HH:MM:SS": int(parsed_dates["txn_datetime"].notna().sum()),
    "Распознано YYYY-MM-DD без времени": int(txn_date_only.notna().sum()),
    "Не распознано после двух форматов": int(txn_calendar.isna().sum())
}, name="Строк")
display(format_summary.to_frame())
print("Примеры дат без времени:", txn_text[date_only].drop_duplicates().head(3).tolist())
print("Фактический диапазон:", txn_calendar.min(), "—", txn_calendar.max())

,Строк
Распознано YYYY-MM-DD HH:MM:SS,566960
Распознано YYYY-MM-DD без времени,49237
Не распознано после двух форматов,0


Примеры дат без времени: ['2024-01-01', '2024-01-02', '2024-01-03']
Фактический диапазон: 2023-11-23 10:48:18 — 2026-07-26 11:47:38


Для оставшихся значений отдельно проверили точную запись `YYYY-MM-DD` и прочитали её как календарную дату. Затем объединили два временных результата распознавания в `txn_calendar`, которая нужна для подсчётов по дням и месяцам. Здесь `fillna` берёт результат чтения той же исходной строки другим форматом. Неизвестная дата при этом не придумывается, а исходный столбец операций остаётся текстовым и неизменным.

Полный формат распознал 566960 строк, формат только с датой — 49237. После их объединения непрочитанных дат не осталось. Общий диапазон начинается 23.11.2023 и заканчивается 26.07.2026. При чтении даты без времени появляется техническое значение `00:00:00`, но оно не сообщает, что платёж действительно произошёл в полночь. Для будущей подготовки нужно сохранить признак отсутствия точного времени; для месячных подсчётов известной календарной даты достаточно.

In [25]:
period_summary = pd.Series({
    "До 2024-01-01": int(txn_calendar.lt("2024-01-01").sum()),
    "2024-01-01 — 2024-12-31": int((txn_calendar.ge("2024-01-01") & txn_calendar.lt("2025-01-01")).sum()),
    "2025-01-01 — 2025-06-30": int((txn_calendar.ge("2025-01-01") & txn_calendar.lt("2025-07-01")).sum()),
    "После 2025-06-30": int(txn_calendar.ge("2025-07-01").sum()),
    "Дата не распознана": int(txn_calendar.isna().sum())
}, name="Строк")
display(period_summary.to_frame())
print("Контроль суммы:", period_summary.sum(), "из", len(transactions_raw), "строк")

,Строк
До 2024-01-01,4
2024-01-01 — 2024-12-31,442820
2025-01-01 — 2025-06-30,173288
После 2025-06-30,85
Дата не распознана,0


Контроль суммы: 616197 из 616197 строк


В этой ячейке разделили все строки операций по отношению к исследуемым периодам. Отдельно посчитали операции до 2024 года, в течение 2024 года, в первом полугодии 2025 года и после 30 июня 2025 года. Верхняя граница основного интервала задана как дата меньше `2025-07-01`, поэтому в него входит весь день 30 июня, включая операции со временем позже полуночи. В конце сложили полученные количества и сверили их с общим числом строк.

В исследуемом интервале находится 616108 строк: 442820 относятся к 2024 году и 173288 — к январю–июню 2025 года. За его пределами 89 строк, из которых 4 находятся до 2024 года, а 85 — после июня 2025 года. Дат, которые не удалось распознать, нет, и сумма всех частей совпадает с 616197 исходными строками. Для дальнейшего сравнения строки вне периода не должны попадать в аналитическую выборку. Сейчас они сохраняются, а само нахождение за пределами задачи не доказывает ошибку исходной даты.

In [26]:
# Считаем количество операций по годам, сортируем по году и называем результат «Строк».
yearly_counts = txn_calendar.dt.year.value_counts().sort_index().rename("Строк")

# Преобразуем даты в месяцы (YYYY-MM), считаем операции и сортируем по времени.
monthly_counts = txn_calendar.dt.to_period("M").value_counts().sort_index()

# Создаём полный список месяцев от самой ранней до самой поздней даты.
calendar_months = pd.period_range(txn_calendar.min(), txn_calendar.max(), freq="M")

# Добавляем отсутствующие месяцы с нулём операций.
monthly_counts = monthly_counts.reindex(calendar_months, fill_value=0).rename("Строк")

# Выводим таблицу количества операций по годам.
display(yearly_counts.rename_axis("Год").to_frame())

# Временно разрешаем pandas показывать до 40 строк таблицы.
with pd.option_context("display.max_rows", 40):
    # Выводим таблицу количества операций по месяцам.
    display(monthly_counts.rename_axis("Месяц").to_frame())

,Строк
Год,
2023,4
2024,442820
2025,173337
2026,36


,Строк
Месяц,
2023-11,2
2023-12,2
2024-01,26578
2024-02,32904
2024-03,37903
2024-04,36992
2024-05,33958
2024-06,36739
2024-07,37831


Здесь посмотрели распределение количества строк по календарным годам и месяцам, чтобы проверить покрытие всего найденного диапазона. Для месячной таблицы создан полный календарный список от первой до последней даты. `reindex` используется только для этой вспомогательной сводки и позволяет показать месяц даже в случае отсутствия строк операций за него. Исходные транзакции и их даты при этом не дополняются и не изменяются.

В каждом из 18 месяцев исследуемого периода есть операции. Это подтверждает наличие записей по всем месяцам, но не доказывает, что источник передал все операции клиентов. В 2024 году наибольшее число строк приходится на декабрь — 46809. Из 85 поздних строк 49 относятся ко второму полугодию 2025 года, ещё 36 — к 2026 году. Различия месячных численностей пока не трактуем как изменение клиентской активности, поскольку периоды имеют разную длину, а повторы и статусы ещё не обработаны.

#### Когда происходили операции клиентов вне справочника

In [27]:
period_masks = {
    "До 2024": txn_calendar.lt("2024-01-01"),
    "2024": txn_calendar.ge("2024-01-01") & txn_calendar.lt("2025-01-01"),
    "Январь–июнь 2025": txn_calendar.ge("2025-01-01") & txn_calendar.lt("2025-07-01"),
    "После 30.06.2025": txn_calendar.ge("2025-07-01")
}
unknown_period_rows = []
for label, mask in period_masks.items():
    selected = unknown_client & mask
    unknown_period_rows.append({"Период": label, "Строк": int(selected.sum()),
                                "Уникальных client_id": transactions_raw.loc[selected, "client_id"].nunique()})
unknown_period_summary = pd.DataFrame(unknown_period_rows).set_index("Период")
display(unknown_period_summary)
inside_period = txn_calendar.ge("2024-01-01") & txn_calendar.lt("2025-07-01")
print("Строк вне справочника в исследуемом периоде:", (unknown_client & inside_period).sum())
print("Строк клиентов справочника в исследуемом периоде:", (~unknown_client & inside_period).sum())
print("Клиентов справочника без операций в исследуемом периоде:",
      (~client_ids.isin(transactions_raw.loc[inside_period, "client_id"])).sum())

,Строк,Уникальных client_id
Период,,
До 2024,2,2
2024,26401,332
Январь–июнь 2025,1740,162
После 30.06.2025,6,6


Строк вне справочника в исследуемом периоде: 28141
Строк клиентов справочника в исследуемом периоде: 587967
Клиентов справочника без операций в исследуемом периоде: 10


В этой проверке объединили два уже известных признака каждой строки: её календарный период и наличие клиента в справочнике. Для каждого периода посчитали строки операций отсутствующих клиентов и число их уникальных ID. Исходную таблицу не ограничивали; временные условия используются только для подсчётов, которые помогают определить влияние охвата источников на будущую выборку.

В 2024 году у клиентов вне справочника 26401 строка, в январе–июне 2025 года — 1740, до 2024 года — 2, после 30.06.2025 — 6. Внутри исследуемого интервала остаётся 28141 строка всех 332 отсутствующих клиентов. Поэтому одно ограничение по датам не устраняет различие между составом таблиц. Уникальные ID из разных периодов нельзя складывать, поскольку один человек может встречаться в нескольких периодах.

Для исследования действующих клиентов нужно использовать ID справочника после решения вопроса повторов. Операции остальных клиентов не войдут в такую выборку, при этом их отсутствие в справочнике само по себе не делает операции ошибочными. У клиентов справочника внутри периода 587967 исходных строк операций, пока с повторениями. Те же 10 клиентов не имеют операций ни в исследуемом интервале, ни во всей выгрузке. Сведения о причине их отсутствия и о статусе клиентов вне справочника в данных не представлены.

#### Согласованность клиентских дат

In [28]:
birth_date = parsed_dates["birth_date"]
registration_date = parsed_dates["registration_date"]
card_open_date = parsed_dates["card_open_date"]
login_before_registration = (parsed_dates["app_last_login"].dt.to_period("M")
                             < registration_date.dt.to_period("M"))
date_logic = pd.Series({
    "Регистрация раньше рождения": int(registration_date.lt(birth_date).sum()),
    "Открытие карты раньше регистрации": int(card_open_date.lt(registration_date).sum()),
    "Рождение после даты снимка": int(birth_date.gt("2025-06-30").sum()),
    "Регистрация после даты снимка": int(registration_date.gt("2025-06-30").sum()),
    "Открытие карты после даты снимка": int(card_open_date.gt("2025-06-30").sum()),
    "Месяц входа раньше месяца регистрации": int(login_before_registration.sum())
}, name="Строк")
display(date_logic.to_frame())
print("Уникальных клиентов с входом до регистрации:",
      clients_raw.loc[login_before_registration, "client_id"].nunique())
display(clients_raw.loc[login_before_registration, ["client_id", "registration_date", "app_last_login"]].head(5))

,Строк
Регистрация раньше рождения,0
Открытие карты раньше регистрации,0
Рождение после даты снимка,0
Регистрация после даты снимка,0
Открытие карты после даты снимка,0
Месяц входа раньше месяца регистрации,21


Уникальных клиентов с входом до регистрации: 21


,client_id,registration_date,app_last_login
47,C000740,06.06.2024,2024-04
652,C000198,04.01.2025,2024-05
724,C001118,17.10.2024,2024-04
785,C005567,06.10.2024,2024-06
1160,C004862,22.04.2024,2024-01


Здесь сравнили связанные клиентские даты, поскольку правильный формат записи ещё не гарантирует правильную последовательность событий. Проверили, не происходит ли регистрация раньше рождения, открытие карты раньше регистрации, а также не выходят ли даты рождения, регистрации и открытия карты за дату снимка. Для последнего входа сравниваются именно календарные месяцы, потому что точный день входа в справочнике неизвестен. Это позволяет не создавать ложное нарушение при регистрации и входе в течение одного месяца.

Регистраций раньше рождения, открытий карты раньше регистрации и перечисленных клиентских дат после снимка не найдено. При этом у 21 строки с 21 уникальным клиентом месяц входа предшествует месяцу регистрации. Например, у `C000198` указана регистрация `04.01.2025`, а последний вход относится к `2024-05`. Эти значения успешно распознаны, поэтому смена формата не устраняет расхождение. По данным нельзя выбрать правильную дату или установить, фиксируют ли поля разные события. При очистке нужно сохранить даты с отметкой о несогласованности и уточнить их смысл у источника, а при отсутствии уточнения отдельно проверить влияние этих клиентов на выводы по последнему входу.

In [29]:
client_date_lookup = (clients_raw[["client_id", "registration_date", "card_open_date"]]
                      .drop_duplicates().set_index("client_id"))
print("Даты однозначны по client_id:", client_date_lookup.index.is_unique)
registration_for_txn = pd.to_datetime(transactions_raw["client_id"].map(client_date_lookup["registration_date"]),
                                    format="%d.%m.%Y", errors="coerce")
card_open_for_txn = pd.to_datetime(transactions_raw["client_id"].map(client_date_lookup["card_open_date"]),
                                 format="%d.%m.%Y", errors="coerce")
before_registration = txn_calendar.dt.normalize().lt(registration_for_txn)
before_current_card = txn_calendar.dt.normalize().lt(card_open_for_txn)
print("Строк с известной датой регистрации клиента:", registration_for_txn.notna().sum())
print("Операций раньше регистрации:", before_registration.sum())
print("Операций раньше открытия текущей карты:", before_current_card.sum())
display(transactions_raw.loc[before_current_card, ["txn_id", "client_id", "txn_datetime"]]
        .assign(card_open_date=transactions_raw["client_id"].map(client_date_lookup["card_open_date"])))

Даты однозначны по client_id: True


Строк с известной датой регистрации клиента: 588048
Операций раньше регистрации: 0
Операций раньше открытия текущей карты: 1


,txn_id,client_id,txn_datetime,card_open_date
275872,990213,C001668,2024-08-26 07:24:00,14.09.2024


В этой ячейке сопоставили даты операций с датами регистрации клиента и открытия его текущей карты. Для сопоставления взяли из справочника только ID и эти две даты. Хотя у 14 клиентов расходятся город и часть сегментов, рассматриваемые даты у них совпадают, поэтому после временного удаления одинаковых сочетаний каждому ID соответствует одна пара дат. Это проверено в выводе `Даты однозначны по client_id: True`. Через `map` каждому платежу подставляется соответствующая дата для проверки; строки операций при этом не размножаются.

Сравнение возможно для 588048 исходных строк, клиент которых есть в справочнике. Среди них операций раньше регистрации в банке не найдено. Одна операция `990213` клиента `C001668` от 26.08.2024 произошла раньше указанного открытия текущей карты 14.09.2024. В источнике говорится именно о текущей карте, а история карт и идентификатор карты в операции отсутствуют. Поэтому платёж мог относиться к прежней карте, и считать его ошибочным пока нельзя. Не следует использовать дату открытия текущей карты как автоматическую границу всей истории клиента; эту строку нужно сохранить с отметкой для проверки.

### Контроль неизменности исходных данных

In [30]:
integrity_rows = []
for name, df in raw_tables.items():
    filename = name + ".csv"
    file_hash = hashlib.sha256((raw_dir / filename).read_bytes()).hexdigest()
    data_hash = hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()
    integrity_rows.append({"Таблица": name, "CSV не изменён": file_hash == source_hashes[filename],
                           "raw не изменён": data_hash == raw_fingerprints[name]})
display(pd.DataFrame(integrity_rows))
assert all(row["CSV не изменён"] and row["raw не изменён"] for row in integrity_rows)


,Таблица,CSV не изменён,raw не изменён
0,clients,True,True
1,transactions,True,True


В конце первого блока повторно рассчитали контрольные отпечатки файлов и исходных таблиц. SHA-256 сравнивает содержимое каждого CSV с тем, которое было прочитано в начале работы. Отдельный отпечаток значений строк и индекса сравнивает текущее состояние `clients_raw` и `transactions_raw` с состоянием сразу после загрузки. Эти две проверки нужны, поскольку файл на диске и таблица в памяти проверяются независимо.

Для обеих таблиц получено `True` в колонках `CSV не изменён` и `raw не изменён`. Значит, первичная диагностика сохранила исходные файлы и исходные значения в памяти. Словари написаний, пробное чтение чисел и дат, временные таблицы без одинаковых копий использовались только для проверок. Реальная очистка и исключение строк не выполнялись; соответствующие правила собраны ниже как план следующего этапа, а `cleaning_log.csv` на этом блоке не создаётся.

### 2.7. Что выявлено

Все количества ниже относятся к исходным данным до очистки: **5723 строки клиентов и 616197 строк операций**. Строки, уникальные клиенты и уникальные операции считаются отдельно. Находки могут пересекаться, поэтому их количества не складываются. Действия в последнем столбце пока только предлагаются; никакая очистка не выполнена.

#### А. Проблемы качества данных

Здесь собраны подтверждённые повторы, различия написаний, неполные и противоречивые сведения, а также подозрительные случаи, требующие проверки. Для подозрительных случаев отдельно указано, что именно наблюдается, какое объяснение возможно и чего пока нельзя установить. Наличие строки в этом разделе не означает, что правильное значение уже известно.

| Проблема или случай для проверки | Что подтверждено данными | Почему важно и что пока неизвестно | Предлагаемая обработка и проверка |
|---|---|---|---|
| **Полные копии записей клиентов** | У 41 `client_id` есть по две полностью одинаковые строки: всего 82 строки, из них 41 лишняя копия. Совпадают все 16 полей. | При объединении с операциями каждая копия клиента повторит его операции и увеличит рассчитанную активность. | Оставить по одной строке из каждой полностью одинаковой пары. Удалить именно 41 повторную копию; разные строки одного клиента этим правилом не обрабатывать. |
| **У одного клиента указаны разные город и сегмент** | Подтверждено: после отделения полных копий остаются 28 строк у 14 клиентов. У всех 14 различается `city`, у 9 из них также `segment`; остальные поля совпадают. После объединения написаний города все 14 расхождений сохраняются. Например, `C000470`: Москва / Екатеринбург и Массовый / Премиальный. | Возможны разные версии сведений или ошибка записи, но причина не установлена. Даты обновления строки нет, поэтому правильный город и сегмент выбрать нельзя. Произвольный выбор изменит размеры и состав клиентских групп. | Сначала уточнить значения у источника. Если это невозможно, рассмотреть сохранение совпадающих полей с отдельной отметкой о неустановленном городе у 14 и сегменте у 9 клиентов. Сравнить размеры, доли и состав групп при отдельном учёте неустановленных значений и при исключении спорных клиентов только из соответствующего разреза. Выбранную стратегию обосновать результатом сравнения; не брать первую или последнюю строку произвольно. |
| **Полные копии операций** | У 1306 `txn_id` есть по две полностью одинаковые строки: всего 2612 строк, из них 1306 лишних копий. Совпадают все 9 полей. | Одна операция может быть посчитана дважды как по количеству, так и по сумме. | Оставить одну строку в каждой полностью одинаковой паре, удалив 1306 повторных копий. Это отдельное правило от повторов с разной точностью даты. |
| **Одна операция продублирована с разной точностью даты** | У 194 `txn_id` остаются 388 различающихся строк. В каждой паре различается только `txn_datetime`: одна строка содержит дату, другая — дату с временем. Во всех 194 парах день распознан, совпадает, обе строки ровно по одной; остальные поля одинаковы. | Это техническое повторение одной операции, а не две разные траты. Выбор строки только с датой потеряет имеющееся точное время. | Для каждого из этих 194 ID оставить строку с указанным временем, убрав вторую строку только с датой. Правило подтверждено для всех пар и не требует восстановления неизвестных значений. |
| **Один город записан несколькими способами** | В `clients.city` 17 написаний обозначают 6 городов. Явный словарь меняет текст у 1622 строк; в их числе 163 строки со значением ` москва`. Примеры: `г. Москва`, `МОСКВА`, ` москва` → `Москва`; `СПб`, `Санкт Петербург`, `г. Санкт-Петербург` → `Санкт-Петербург`; `Н. Новгород` → `Нижний Новгород`. Полный словарь с частотами приведён выше. | Без объединения один город будет разбит на несколько групп. Простого изменения регистра недостаточно для сокращений и префикса `г.`. | Применить приведённый явный словарь к перечисленным вариантам. Остальные значения не менять автоматически. Это устранит различия написаний, но не противоречия у 14 клиентов из отдельной строки таблицы. |
| **Одна категория трат записана несколькими способами** | `transactions.category` содержит 12 написаний для 9 категорий. Для 51025 строк требуется смена написания: `Связь и Интернет` — 19354 строки; `кафе и рестораны` — 15920; `Кафе/рестораны` — 15751. У вариантов связи преобладает MCC `4814`, у вариантов кафе — `5812`, как и у основных написаний. | Оборот и число операций одной категории будут разделены между разными названиями. Разница 12 против 9 объясняется именно тремя дополнительными написаниями. | Заменить только эти три варианта по словарю: на `Связь и интернет` и `Кафе и рестораны`. Не переписывать другие поля. Согласованность конкретной категории с MCC проверять отдельно. |
| **Пары категории и MCC, требующие проверки** | После временного объединения написаний подтверждены 1569 строк с 1562 уникальными `txn_id`, у которых пара `category` / `mcc_code` отличается от преобладающей пары этого MCC в выгрузке. Пример: `txn_id = 100210`, `Одежда и обувь` и `5411`; наиболее частая категория для `5411` здесь — `Продукты`. | Преобладание пары не доказывает правильность её применения к каждой операции. Возможны ошибка категории, ошибка MCC или особенность классификации. По данным неизвестно, какое поле нужно менять и нужно ли менять его вообще. | Сохранить оба исходных значения и отдельно отметить несовпадение. Уточнить соответствие у источника. Если уточнение недоступно, сравнить размеры и месячную динамику категорий с исходными категориями и с отдельным учётом спорных строк вне обычных категорий. Не исправлять по большинству автоматически и не удалять эти операции из всех метрик активности; выбор стратегии описать как допущение. |
| **Доход не заполнен** | `clients.monthly_income` пуст у 514 из 5723 строк (8,98%). Это пустые строки, сохранённые при загрузке, а не нули. | Доход этой части клиентов неизвестен. Удаление клиентов или заполнение одним средним без исследования изменит состав данных и распределение дохода. | Признать отсутствие значения в рабочем числовом поле, сохранить признак исходной пустоты. Не удалять клиента и не подставлять доход автоматически; решение о заполнении обосновать на отдельном этапе пропусков. |
| **Подозрительное повторяющееся значение дохода `-999.0`** | В 57 строках у 56 уникальных клиентов `monthly_income` равен ровно `-999.0`. Это все отрицательные доходы; текст успешно читается как число. Назначение такого значения в описании отсутствует. | Точное повторение одного отрицательного числа в поле месячного дохода похоже на специальный код. Это возможное объяснение, а не подтверждённое отсутствие дохода. Автоматическая замена пропуском или признание суммы реальным доходом одинаково требуют обоснования. | Уточнить значение у источника. При подтверждённом коде отсутствия предложить замену в рабочем поле с отдельным признаком и журналом. Если подтверждения нет, сохранить исходное число и отметку о неопределённой трактовке; перед выбором обработки сравнить выводы по доходу с отдельным учётом этих строк и без их участия в числовых расчётах. Не заполнять нулём или средним автоматически. |
| **Возможное различие единиц дохода** | При просмотре наименьших фактических значений `monthly_income` найдены 184 строки с 182 уникальными клиентами в диапазоне `0 < доход < 1000`: от 26,5 до 548,6 при заявленной единице «рубли». Наименьшее положительное значение вне этого диапазона — 17300. | Замечен разрыв масштаба. Возможная причина — запись части сумм в тысячах рублей, но подтверждения нет; малые суммы могут иметь другое объяснение. Порог 1000 служит только для описания замеченного диапазона и не доказывает ошибку любой строки. | Проверить единицы измерения у источника. Не умножать на 1000, не удалять и не заменять пропусками автоматически. До уточнения сохранить исходные числа и отдельно проверить влияние этого диапазона на выводы по доходу; перевод единиц допустим только как подтверждённое правило либо явно обозначенный сценарий чувствительности. |
| **Оценка сервиса известна не для всех** | `clients.satisfaction_score` пуст у 3431 строки (59,95% от 5723); непустые оценки принимают значения от 1 до 5. | Оценки описывают только ответившую часть базы. Пустое значение не означает низкую оценку, а порядковую шкалу нельзя автоматически трактовать как обычную числовую сумму. | Сохранить отсутствие ответа отдельным признаком; не заменять его нулём, средней оценкой или модой без обоснования. Анализировать оценки с указанием числа ответов, а механизм пропусков исследовать отдельно. |
| **Неуточнённая трактовка `Не указан` в источнике зарплаты** | В фактическом наборе из трёх значений `clients.income_source` категория `Не указан` встречается у 1476 строк (25,79% от 5723). Из них у 1095 доход положительный, у 370 доход пустой, у 11 записано отрицательное число. | По названию источник зарплаты, вероятно, не уточнён, но процесс ввода категории неизвестен. Само её наличие не является доказанной ошибкой. Она не означает автоматически пустой доход, отсутствие зарплаты или безработицу; это подтверждается наличием положительного дохода у 1095 строк. | До уточнения хранить как исходную отдельную категорию. Не заменять её пропуском или другим источником автоматически и не переносить её трактовку на `monthly_income`. Если позже будет принято решение считать её отсутствием сведений об источнике, отдельно описать это решение и проверить, как оно меняет размеры и доли групп по источнику зарплаты. |
| **Вход в приложение раньше регистрации в банке** | У 21 строки с 21 уникальным клиентом `app_last_login` предшествует месяцу `registration_date`. Например, `C000198`: регистрация `04.01.2025`, последний вход `2024-05`. Сравнивались месяцы, поэтому неизвестный день входа не создаёт ложного нарушения внутри одного месяца. | Приведение форматов не устраняет расхождение. Неизвестно, неверна ли одна дата или поля фиксируют разные события; по ним нельзя уверенно судить о последней цифровой активности. | Сохранить даты и признак несогласованности, уточнить смысл полей и значения у источника. Не сдвигать даты автоматически. Если уточнения нет, отдельно проверить влияние этих 21 клиентов на выводы по последнему входу. |

#### Б. Особенности данных и ограничения анализа

Эти условия влияют на расчёт и интерпретацию, но сами по себе не подтверждают ошибку. Например, возврат может иметь отрицательную сумму, а операция до открытия текущей карты может относиться к прежней карте. Ограничения периода, охвата и точности нужно учитывать явно, сохраняя различие между исключением из конкретного анализа и исправлением ошибочной записи.

| Особенность или ограничение | Что наблюдается | Как влияет на анализ | Что предусмотреть при подготовке и расчётах |
|---|---|---|---|
| **Операции клиентов, которых нет в справочнике** | В `transactions` 332 клиента отсутствуют в `clients`; их `client_id` заполнены. На них приходится 28149 строк: 26401 в 2024 году, 1740 в январе–июне 2025, 2 до 2024 года и 6 после 30.06.2025. В исследуемом интервале остаётся 28141 строка всех 332 клиентов. | Ограничение датами не устраняет эти операции. Без клиентской записи неизвестны сегмент, город, согласие и принадлежность клиента к действующей базе на дату снимка. Отсутствие в справочнике не доказывает ошибку или закрытие клиента. | Для исследования действующих клиентов брать ID из справочника после решения вопроса повторов; 28141 исходная строка внутри периода не попадёт в эту клиентскую выборку. Не заполнять клиентские признаки и согласие предположениями. Сохранить сводку исключённого охвата, не удаляя сырьевые файлы. |
| **У клиентов справочника нет операций в выгрузке** | У 10 из 5668 уникальных клиентов справочника нет строк операций ни во всей выгрузке, ни в исследуемом периоде. | При объединении только по имеющимся операциям такие клиенты исчезнут. Отсутствие строк не позволяет различить неактивность и неполный охват источника. | Сохранить их при построении клиентской таблицы через присоединение операций к справочнику. Отдельно отметить отсутствие наблюдаемых операций; нулевые счётчики трактовать только как отсутствие операций в предоставленной выгрузке. Не объявлять снижением активности без проверяемой базы сравнения. |
| **Месяц входа отсутствует у клиентов без приложения** | `app_last_login` пуст у 2180 строк (38,09% от 5723). Все эти строки имеют `has_mobile_app = Нет`; у всех 3543 строк с `Да` месяц входа указан. Несовпадений этой пары признаков нет. | В текущем снимке отсутствие месяца согласуется с отсутствием установленного приложения. Это не доказывает, что человек никогда не устанавливал приложение. | Сохранить отсутствие месяца и признак `has_mobile_app`. Не заполнять дату последнего входа вымышленным месяцем и не считать такую пустоту обычным техническим сбоем. Историю установок из этих двух полей не восстанавливать. |
| **Для части операций неизвестно точное время** | В `txn_datetime` 566960 строк имеют формат `YYYY-MM-DD HH:MM:SS`, 49237 — `YYYY-MM-DD`. Два явных формата распознают все строки; 194 повторные строки только с датой разобраны выше отдельно. | Один ожидаемый формат теряет 49237 дат. Техническое время `00:00:00` при чтении даты нельзя выдавать за известное время платежа; анализ по часам для этой части строк невозможен. | Привести даты с явным учётом обоих форматов, сохранив исходный текст или признак отсутствия времени. Для анализа по месяцам использовать календарную дату; часы изучать только там, где время задано в источнике. Не восстанавливать его предположениями. |
| **Часть операций вне периода исследования** | Фактический диапазон — 23.11.2023–26.07.2026. Вне периода 01.01.2024–30.06.2025 находятся 89 строк: 4 до 2024 года и 85 после июня 2025 (49 во втором полугодии 2025 и 36 в 2026). Внутри периода 616108 исходных строк. | Даже правильно распознанная дата может не относиться к задаче. Эти строки нельзя добавлять к одному из сравниваемых периодов; происхождение поздних дат неизвестно. | Для аналитической выборки применить `>= 2024-01-01` и `< 2025-07-01`, сохранив сведения об исключении. Пересчитать затронутые строки после технической дедупликации и не складывать эту численность с другими исключениями без учёта пересечений. Не объявлять дату ошибочной только из-за выхода за рамки задачи. |
| **Операция раньше открытия текущей карты** | Одна строка: `txn_id = 990213`, клиент `C001668`, операция `26.08.2024`, текущая карта открыта `14.09.2024`. Среди 588048 строк с клиентом в справочнике операций раньше регистрации в банке нет. | Справочник содержит дату текущей карты, а не всю историю карт. Операция могла относиться к прежней карте; в таблице операций нет ID карты для проверки. Это не подтверждённая ошибка. | Сохранить операцию с отметкой для проверки. Не использовать дату открытия текущей карты как автоматический фильтр всей истории операций или начало наблюдения клиента; при необходимости запросить историю карт. |
| **Разные статусы и знак суммы** | Успешных строк — 596679, отклонённых — 12114, возвратов — 7404. Все возвраты имеют отрицательную сумму; все успешные и отклонённые строки — положительную. Нулевых сумм нет. | Отрицательная сумма возврата сама по себе корректна, а положительная сумма отказа не означает состоявшуюся покупку. Подсчёт всех строк как оплат даст другой смысл активности. | Перед расчётом метрики явно определить учёт успешных оплат, отказов и возвратов. Не удалять возвраты как отрицательные ошибки и не считать отклонённые операции успешными списаниями. Не пытаться восстановить связь возврата с исходной покупкой без соответствующего поля. |
| **Суммы в разных валютах** | В `currency` три значения: RUB — 613688 строк, EUR — 1275, USD — 1234. Итого 2509 строк в EUR/USD; курса пересчёта в двух источниках нет. | Сумма рублей, долларов и евро не имеет единой денежной единицы. Одинаково записанное число не делает операции сопоставимыми по сумме. | Если денежная метрика потребуется, считать суммы отдельно по валютам или использовать документированный источник курсов и правило даты конвертации. Без курсов не подменять валюту и не переводить суммы условным коэффициентом; валютные операции не удалять только из-за валюты. |
| **Числовые и временные поля пока хранятся текстом** | Все 16 + 9 полей загружены как `string`. В двух денежных полях нет нечисловых непустых значений и бесконечностей; `amount` использует десятичную запятую, `monthly_income` — точку. Клиентские даты используют ISO, `DD.MM.YYYY` и `YYYY-MM`; идентификатор отделения имеет вид `00395`. | Для расчётов нужны числа и даты, но превращение кодов отделений или ID в числа потеряет смысл и ведущие нули. Успешное чтение числа не заменяет проверку `-999.0` и статуса операции. | В рабочем наборе явно преобразовать только количественные и временные поля, контролируя новые пропуски. Идентификаторы оставить строками; месячный вход сохранить с месячной точностью. Все реальные преобразования типов и специальных значений описать в журнале. |
| **Периоды разной длины — условие анализа** | Базовый период — 12 месяцев 2024 года, сравниваемый — 6 месяцев 2025 года. Записи есть в каждом из 18 месяцев, но это не доказательство полноты сбора. | Сравнение общих чисел операций или сумм за 12 и 6 месяцев создаст видимое падение даже без изменения интенсивности использования. Сезонность тоже может влиять на сравнение. | На этапе выбора метрики считать активность в пересчёте на месяц и проверить сопоставимые календарные месяцы. Здесь метрика и пороги снижения ещё не рассчитывались. |
| **Характеристики клиента даны снимком, без истории** | `clients` по заданию отражает действующих клиентов на 30.06.2025. В таблице нет дат изменения города, сегмента, дохода, согласия или установки приложения; всего 5668 уникальных ID, включая неразрешённые разные записи у 14 клиентов. | Нельзя доказать, что текущая характеристика была такой же в 2024 году, или выбрать последнюю из двух записей по порядку строк. Связь текущего сегмента со снижением не является доказанной причиной снижения. | Использовать признаки как состояние на дату снимка, указывать это в интерпретации и не создавать вымышленную историю. Для разрешения разных записей и утверждений об изменении признаков нужен источник с датами обновления. |

Пустых ключей и пробелов по краям ID нет. В остальных категориальных полях не обнаружены дополнительные варианты, устраняемые регистром или краевыми пробелами. Все непустые даты распознаны при учёте двух форматов операций; регистраций раньше рождения и открытий карты раньше регистрации нет. Отсутствие месяца входа согласуется с текущим признаком отсутствия приложения. Это результаты конкретных проверок, а не подтверждение полноты или достоверности всех сведений.

Фактические значения исследовались по частотам категорий, результатам чтения чисел и дат и просмотру нижней части диапазона дохода. `Не указан`, `-999.0` и малые положительные доходы не заменялись пропусками. Причины пустого дохода и отсутствующих оценок, назначение специального числа и возможное различие единиц ещё требуют отдельного выяснения. Механизмы MCAR/MAR/MNAR, распределения и полноценный анализ выбросов в этом блоке не исследуются.

Правильный город и часть сегментов у 14 клиентов, а также правильность отдельных полей в 1569 редких парах категории/MCC по имеющимся данным не определены. Предложенные способы учёта таких строк являются вариантами, а не восстановлением истины. Если уточнение источника недоступно, до принятия стратегии нужно сравнить относящиеся к ней размеры групп, доли и месячную динамику, а затем описать выбранное допущение и его влияние на выводы. Стратегии проверки только запланированы: сейчас метрика активности и группы кампании не рассчитывались.

На следующем этапе можно начать с подтверждённых полных копий и пар разной точности операций, после чего отдельно выбрать правила для неоднозначных записей и значений. Условия периода, клиентского охвата, статусов, валют и точности дат учитываются как условия конкретного расчёта. Количество строк каждого действия нужно пересчитать с учётом предыдущих правил и пересечений; справочник должен содержать одну запись на клиента до объединения с операциями. Клиентов без наблюдаемых операций нельзя терять при построении клиентской таблицы.

SHA-256 обоих исходных CSV и отпечатки `clients_raw` / `transactions_raw` проверяются последней кодовой ячейкой. Реальных преобразований данных не было, поэтому `cleaning_log.csv` не создаётся. Каждое будущее изменение оформляется по схеме `field, issue, scope, n_rows, value_before, action, value_after, rationale` с фактическим числом затронутых строк и обоснованием. Завершённый первый блок фиксируется отдельным Git-коммитом после проверки воспроизводимости.
